In [2]:
# figure3.py
# Figure 3 (main) and Figure S1 (supplement): coverage against mean set size for the six decision rules.
# Runs on its own. Reads modeling_table.csv and, from core_outputs/ (glioma_cp_core.py):
#   egd_probas.npy, utsw_probas.npy (section 2), is_test_egd.npy, is_test_utsw.npy (section 4), decision_rules_200_splits.csv (section 5)
# Writes core_outputs/figure3_curves.csv, Figure3.png/.tiff, FigureS1.png
# Set rules are run at targets 80 to 95% (probability sets, single-threshold, Mondrian), and abstention at shares of
# single-subtype calls from 100% to 0%, so each traces a line. Everything is fitted on the calibration half exactly as in
# section 5. The two argmax rules are single points taken from section 5 (they always give one subtype, set size 1).
# Figure 3: coverage of the least-covered subtype against mean set size over all patients, one panel per cohort.
# Figure S1: coverage against set size for each subtype and for all patients.

import os
import numpy as np
from itertools import product
import pandas as pd
from sklearn.linear_model import LogisticRegression
from crepes import ConformalClassifier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUT = 'core_outputs'
table = pd.read_csv('modeling_table.csv')
y_egd = table[table['site'] == 'EGD']['2021_glioma_class'].values
y_utsw = table[table['site'] == 'UTSW']['2021_glioma_class'].values
classes = np.unique(table[table['site'].isin(['TCGA', 'UCSF'])]['2021_glioma_class'].values)
cohorts = {'EGD': (y_egd, np.load(os.path.join(OUT, 'egd_probas.npy'))),
           'UTSW': (y_utsw, np.load(os.path.join(OUT, 'utsw_probas.npy')))}
is_test = {'EGD': np.load(os.path.join(OUT, 'is_test_egd.npy')), 'UTSW': np.load(os.path.join(OUT, 'is_test_utsw.npy'))}
rules = pd.read_csv(os.path.join(OUT, 'decision_rules_200_splits.csv'))
eye3 = np.eye(3, dtype=int)

# ---- curves: every split, every setting ----
curve_levels = [0.80, 0.85, 0.90, 0.95]   # same targets as Figure 2
commit_shares = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.0]   # abstention: share of calibration patients given one subtype

curve_rows = []
for cohort, seed in product(['EGD', 'UTSW'], range(200)):
    y, p = cohorts[cohort]
    true_col = np.searchsorted(classes, y)
    test = np.where(is_test[cohort][seed])[0]
    cal = np.where(~is_test[cohort][seed])[0]
    yt = true_col[test]
    scores = 1 - p
    p_clip = np.clip(p, 1e-6, 1 - 1e-6)
    log_odds = np.log(p_clip / (1 - p_clip))

    # same recalibration and conformal fits as section 5
    recal = np.zeros(p.shape)
    for k in range(3):
        sig = LogisticRegression(C=1e6, max_iter=1000).fit(log_odds[cal, k].reshape(-1, 1), (true_col[cal] == k).astype(int))
        recal[:, k] = sig.predict_proba(log_odds[:, k].reshape(-1, 1))[:, 1]
    recal = recal / recal.sum(axis=1, keepdims=True)
    cc_m = ConformalClassifier()
    cc_m.fit(scores[cal, true_col[cal]], bins=y[cal])
    cc_1 = ConformalClassifier()
    cc_1.fit(scores[cal, true_col[cal]])
    order = np.argsort(-recal[test], axis=1)
    sorted_p = np.take_along_axis(recal[test], order, axis=1)
    top_cal = np.sort(p[cal].max(axis=1))[::-1]                  # calibration top probabilities, highest first

    curve_sets = []
    for level in curve_levels:
        set_prob = np.zeros((len(test), 3), dtype=int)
        np.put_along_axis(set_prob, order, ((np.cumsum(sorted_p, axis=1) - sorted_p) < level).astype(int), axis=1)
        set_single = cc_1.predict_set(scores[test], confidence=level, smoothing=False)
        set_mondrian = np.column_stack([cc_m.predict_set(scores[test], bins=np.full(len(test), cls), confidence=level, smoothing=False)[:, c]
                                        for c, cls in enumerate(classes)])
        curve_sets.append(('Recalibrated probability sets', level, set_prob))
        curve_sets.append(('Single-threshold conformal', level, set_single))
        curve_sets.append(('Mondrian conformal', level, set_mondrian))

    for share in commit_shares:
        n_commit = int(round(share * len(cal)))
        cutoff = top_cal[n_commit - 1] if n_commit > 0 else 1.01
        commit = p[test].max(axis=1) >= cutoff
        set_abstain = np.ones((len(test), 3), dtype=int)
        set_abstain[commit] = eye3[np.argmax(p[test], axis=1)][commit]
        curve_sets.append(('Probability abstention', share, set_abstain))

    for rule, level, s in curve_sets:
        covered = s[np.arange(len(test)), yt] == 1
        size = s.sum(axis=1)
        curve_rows.append({'cohort': cohort, 'seed': seed, 'rule': rule, 'level': level,
                           'cov_A': covered[yt == 0].mean(), 'cov_G': covered[yt == 1].mean(), 'cov_O': covered[yt == 2].mean(),
                           'cov_all': covered.mean(),
                           'size_A': size[yt == 0].mean(), 'size_G': size[yt == 1].mean(), 'size_O': size[yt == 2].mean(),
                           'size': size.mean()})

curves = pd.DataFrame(curve_rows).groupby(['cohort', 'rule', 'level']).mean(numeric_only=True).reset_index()
curves.to_csv(os.path.join(OUT, 'figure3_curves.csv'), index=False)


# ---- argmax points from section 5: mean over splits. Coverage over all patients = per-subtype coverage weighted by test counts ----
counts = []
for cohort in ['EGD', 'UTSW']:
    y = cohorts[cohort][0]
    for seed in range(200):
        test = is_test[cohort][seed]
        counts.append({'cohort': cohort, 'seed': seed, 'n_A': np.sum(y[test] == classes[0]),
                       'n_G': np.sum(y[test] == classes[1]), 'n_O': np.sum(y[test] == classes[2])})
rules = rules.merge(pd.DataFrame(counts), on=['cohort', 'seed'])
rules['cov_all'] = (rules['cov_A'] * rules['n_A'] + rules['cov_G'] * rules['n_G'] + rules['cov_O'] * rules['n_O']) / (rules['n_A'] + rules['n_G'] + rules['n_O'])
points = rules.groupby(['cohort', 'rule']).mean(numeric_only=True).reset_index()
curves['cov_min'] = curves[['cov_A', 'cov_G', 'cov_O']].min(axis=1)        # lowest of the three subtype coverages
points['cov_min'] = points[['cov_A', 'cov_G', 'cov_O']].min(axis=1)

plt.rcParams.update({'font.family': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'font.size': 8})
line_rules = [('Class-conditional conformal', 'Mondrian conformal', '#0072B2', 'o'),
              ('Standard conformal', 'Single-threshold conformal', '#009E73', 's'),
              ('Probability sets', 'Recalibrated probability sets', '#E69F00', '^'),
              ('Probability-based abstention', 'Probability abstention', '#666666', 'D')]
point_rules = [('Recalibrated argmax', 'Recalibrated argmax', '#000000', 'X'),
               ('Class-weighted argmax', 'Locally weighted argmax', '#CC79A7', 'P')]

# ---- Figure 3: lowest subtype coverage against mean set size, one panel per cohort ----
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.6), sharey=True)
for ax, cohort, letter in zip(axes, ['EGD', 'UTSW'], ['A', 'B']):
    for label, rule, color, mark in line_rules:
        d = curves[(curves['cohort'] == cohort) & (curves['rule'] == rule)].sort_values('size')
        ax.plot(d['size'], d['cov_min'], color=color, marker=mark, markersize=3, linewidth=1.2, label=label)
    for label, rule, color, mark in point_rules:
        t = points[(points['cohort'] == cohort) & (points['rule'] == rule)]
        ax.plot(t['size'], t['cov_min'], color=color, marker=mark, markersize=8, markeredgecolor='white', markeredgewidth=0.6, linestyle='none', label=label)
    ax.set_title(letter + '   ' + ('EGD' if cohort == 'EGD' else 'UTSW-Glioma'), loc='left', fontweight='bold', fontsize=9)
    ax.set_xlabel('Mean set size (subtypes per patient)')
    ax.set_xlim(0.9, 3.05)
    ax.set_ylim(0, 1.02)
    ax.grid(color='#eeeeee', linewidth=0.6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
axes[0].set_ylabel('Coverage of the least-covered subtype')
fig.legend(*axes[0].get_legend_handles_labels(), loc='lower center', ncol=3, frameon=False, fontsize=7.5)
plt.tight_layout(rect=[0, 0.13, 1, 1])
plt.savefig(os.path.join(OUT, 'Figure3.png'), dpi=1200)
plt.savefig(os.path.join(OUT, 'Figure3.tiff'), dpi=1200, pil_kwargs={'compression': 'tiff_lzw'})
plt.close(fig)

# ---- Figure S1: the same curves for each subtype and for all patients ----
fig, axes = plt.subplots(2, 4, figsize=(10, 5.6), sharex=True, sharey=True)
panel_cols = [('size_A', 'cov_A', 'Astrocytoma'), ('size_G', 'cov_G', 'Glioblastoma'), ('size_O', 'cov_O', 'Oligodendroglioma'), ('size', 'cov_all', 'All patients')]
for row, cohort in enumerate(['EGD', 'UTSW']):
    for col, (x_col, y_col, name) in enumerate(panel_cols):
        ax = axes[row, col]
        for label, rule, color, mark in line_rules:
            d = curves[(curves['cohort'] == cohort) & (curves['rule'] == rule)].sort_values(x_col)
            ax.plot(d[x_col], d[y_col], color=color, marker=mark, markersize=2.5, linewidth=1.1, label=label)
        for label, rule, color, mark in point_rules:
            t = points[(points['cohort'] == cohort) & (points['rule'] == rule)]
            ax.plot(t['size'], t[y_col], color=color, marker=mark, markersize=7, markeredgecolor='white', markeredgewidth=0.5, linestyle='none', label=label)
        ax.set_title(('EGD' if cohort == 'EGD' else 'UTSW-Glioma') + ': ' + name, loc='left', fontsize=8, fontweight='bold')
        ax.grid(color='#eeeeee', linewidth=0.6)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
for ax in axes[1]:
    ax.set_xlabel('Mean set size')
for ax in axes[:, 0]:
    ax.set_ylabel('Coverage')
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc='lower center', ncol=6, frameon=False, fontsize=7.5)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig(os.path.join(OUT, 'FigureS1.png'), dpi=1200)
plt.close(fig)

findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberati